# Day 3b — Human in the loop

`book_room` writes. So far, nothing has stopped it: the model decides, the tool
fires, a room is booked.

In this notebook the run **pauses** before that tool and waits for a human. To
pause, an agent must be able to come back later — so we start with the thing
that makes coming back possible: the **checkpointer**.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below — e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

### Restore yesterday's world

Every day-3 notebook starts from the same booking agent you built on day 2.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool

HOTELS = {"Hotel Anders":  {"free": True,  "eur": 92},
          "Pension Krumm": {"free": True,  "eur": 61},
          "Grand Mitte":   {"free": False, "eur": 210}}

@tool
def check_availability(hotel: str) -> dict:
    """Live availability and price (EUR/night) for one hotel."""
    return HOTELS.get(hotel, {"error": f"unknown hotel: {hotel}"})

@tool
def list_hotels() -> list:
    """All hotels we can book, cheapest first."""
    return sorted(HOTELS, key=lambda h: HOTELS[h]["eur"])

@tool
def book_room(hotel: str, nights: int) -> dict:
    """Book a room. THIS ONE WRITES."""
    return {"confirmation": f"GIU-{abs(hash(hotel + str(nights))) % 10000:04d}"}

INSTRUCTIONS = ("You are the GIU booking assistant. "
                "Check availability before booking. Prices in EUR.")
print("world restored")

## 1 · Checkpointers — the list, persisted

Day 2's lesson: *the agent remembers nothing — the list does.* You re-sent the
transcript by hand. A **checkpointer** does it for you: it saves the state after
every step, filed under a `thread_id`.

Memory is **storage, not magic**.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model=llm,
    tools=[check_availability, list_hotels, book_room],
    system_prompt=INSTRUCTIONS,
    checkpointer=InMemorySaver())        # ← the new argument

amr  = {"configurable": {"thread_id": "amr"}}
sara = {"configurable": {"thread_id": "sara"}}

agent.invoke({"messages": [HumanMessage("I need a room in Berlin, 2 nights.")]}, amr)
out = agent.invoke({"messages": [HumanMessage("Make it three nights.")]}, amr)
print("thread amr :", out["messages"][-1].content[:150])

out = agent.invoke({"messages": [HumanMessage("How many nights did I ask for?")]}, sara)
print("thread sara:", out["messages"][-1].content[:150])   # clean slate — different thread

Note what you did **not** do: you never passed the old messages back. Same
`thread_id` → it continues. New `thread_id` → a fresh conversation.

One thread = one conversation. Your backend hands out the ids; the agent never
knows.

In [ ]:
# the whole transcript is inspectable — it's data, not vibes
state = agent.get_state(amr)
for m in state.values["messages"]:
    print(m.type.upper().ljust(6), "→", str(m.content)[:80])

In [ ]:
# swap ONE line and conversations survive a restart
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

saver = SqliteSaver(sqlite3.connect("checkpoints.db", check_same_thread=False))

durable = create_agent(model=llm,
    tools=[check_availability, list_hotels, book_room],
    system_prompt=INSTRUCTIONS,
    checkpointer=saver)

durable.invoke({"messages": [HumanMessage("Hold a room at Pension Krumm for me.")]},
               {"configurable": {"thread_id": "amr-durable"}})
print("saved to checkpoints.db")

**Your turn (1)** — restart the kernel. Re-run the setup cells and the SQLite
cell, then ask *"what did I ask you to hold?"* on thread `amr-durable`. The
process died; the conversation did not.

In [ ]:
# after the restart, prove persistence here


## 2 · The write waits

Now the guard. `HumanInTheLoopMiddleware` interrupts the run **before** a named
tool executes. The tool call exists; it just does not run yet.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

guarded = create_agent(model=llm,
    tools=[check_availability, list_hotels, book_room],
    system_prompt=INSTRUCTIONS,
    checkpointer=InMemorySaver(),        # pausing REQUIRES persistence
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={"book_room": True})])   # reads flow, this one waits

cfg = {"configurable": {"thread_id": "guarded-1"}}
out = guarded.invoke({"messages": [HumanMessage(
    "Book me the cheapest available room for 2 nights.")]}, cfg)

state = guarded.get_state(cfg)
print("run is parked before:", state.next)
print("\nwhat it wants to do:")
print(state.tasks[0].interrupts if state.tasks else "no interrupt")

Read that interrupt payload — it is what you would render in a UI: *"the agent
wants to call `book_room(hotel=..., nights=...)`. Approve?"*

The run is now sitting in the checkpointer under `guarded-1`, waiting.

In [ ]:
# a human signs off — the run reloads and continues exactly where it stopped
from langgraph.types import Command

resumed = guarded.invoke(Command(resume=[{"type": "accept"}]), cfg)
print(resumed["messages"][-1].content)

> **If that cell errors:** the resume payload shape varies slightly by version.
> Print `state.tasks[0].interrupts` and match what it asks for. The *concept* is
> stable: reads flow, writes wait for a signature.

**Your turn (2)** — the reject path. Start a fresh thread, get the interrupt,
then resume with `[{"type": "reject"}]` (or the shape your version showed).
What does the agent say to the user? Does it try again, or give up?

In [ ]:
# your reject path here


**Your turn (3)** — over-guard it: add `check_availability` to `interrupt_on`
and run a normal question. Count how many approvals one simple request now
costs you.

Guardrails have a price. Put them where writes happen — **not everywhere.**

In [ ]:
# your over-guarded agent here


---
## Wrap

`checkpointer=` + `thread_id` → conversations that continue and survive restarts ·
`HumanInTheLoopMiddleware` → the run parks before a write and resumes with
`Command(resume=…)` · the pause lives in the checkpointer, which is why the two
belong in the same session.

**Next (3c):** the system prompt is one big string. Time to give the agent a
binder instead.